In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType

catalog_name = 'ecommerce'

In [0]:
df_silver_customers = spark.table(f"{catalog_name}.bronze.brz_customers")

df_silver_customers.printSchema()

In [0]:
row_count, column_count = df_silver_customers.count(), len(df_silver_customers.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
# preview

total = df_silver_customers.count()
distinct_pks = df_silver_customers.select("customer_id").distinct().count()

print(f"Total rows: {total}")
print(f"Distinct PKs: {distinct_pks}")
print(f"Duplicates: {total - distinct_pks}")

fully_distinct = df_silver_customers.distinct().count()

print(f"Fully distinct rows: {fully_distinct}")
print(f"Distinct rows: {distinct_pks}")

## 1. customer_id

In [0]:
# 1.1 transformation: slv_customers -> customer_id

# drop duplicates, then remove null customer_id
df_silver_customers = df_silver_customers.dropDuplicates(["customer_id"])


In [0]:
# 1.2 validation: slv_customers -> customer_id

df_silver_customers.select("customer_id").filter(
    F.col("customer_id").isNull() |
    ~F.col("customer_id").like("CUST%") |
    (F.length(F.col("customer_id")) != 17)
).show()

#duplicates
df_silver_customers.groupBy("customer_id").count() \
    .filter(F.col("count") > 1).show()


## 2. phone

In [0]:
# 2.1 transformation: slv_customers -> phone

df_silver_customers = df_silver_customers.withColumn(
    "phone",
    F.regexp_replace(F.col("phone"), "[^0-9]", ""))

df_silver_customers = df_silver_customers.withColumn(
    "is_phone_valid",
    F.when(F.col("phone").isNull(), F.lit(None))
    .when(
        (F.length(F.col("phone")) >= 10) &
        (F.length(F.col("phone")) <= 15) &
        F.col("phone").rlike("^[0-9]"), True
    )
    .otherwise(False)
)

In [0]:
# 2.2 validation: slv_customers -> phone

invalid_phones = df_silver_customers.filter(
    F.col("phone").isNotNull() &
    (F.length(F.col("phone")) < 10) | (F.length(F.col("phone")) > 15)
)

invalid_phones.show(10)


## 3. country_code

In [0]:
# 3.1 transformation: slv_customers -> country_code
df_silver_customers = df_silver_customers.withColumn(
    "country_code", F.upper(F.trim(F.col("country_code")))
)



In [0]:
# 3.2 validation: slv_customers -> country_code

valid_country_codes = [
    row["country_code"]
    for row in spark.table(f"{catalog_name}.silver.slv_countries_clean")
    .select("country_code")
    .distinct()
    .collect()
]

df_silver_customers.filter(
    ~F.col("country_code").isin(valid_country_codes) | (F.length(F.col("country_code")) != 2)) \
    .select("country_code").distinct().show()


## 4. country_name

In [0]:
# 4.1 transformation: slv_customers -> country_name

from_silver_countries_clean = spark.table(f"{catalog_name}.silver.slv_countries_clean") \
    .select("country_code", "country_name")

df_silver_customers = df_silver_customers.drop("country") \
    .join(from_silver_countries_clean, on = "country_code", how = "left")

In [0]:
# 4.2 validation: slv_customers -> country_name

df_silver_customers.filter(F.col("country_name").isNull()) \
    .select("customer_id", "country_code", "country_name") \
    .show()

## 5. state

In [0]:
# 5.1 transformation: slv_customers -> state
df_silver_customers = df_silver_customers.withColumn(
    "state", F.upper(F.col("state"))
)



In [0]:
# 5.2 validation: slv_customers -> state

df_silver_customers.filter(F.col("state").isNull()) \
    .select("customer_id", "state").show()

## Clean & Quarantined Data

In [0]:
# 6.1 Quarantine Bad Data -> df_silver_customers_clean & df_silver_customers_quarantine

df_silver_customers_clean = df_silver_customers.filter(
  F.col("customer_id").isNotNull() & (F.col("customer_id") != "") &
  F.col("country_code").isNotNull() & (F.col("country_code") != "") & (F.col("country_code").isin(valid_country_codes)) & (F.length(F.col("country_code")) == 2)
)

df_silver_customers_quarantine = df_silver_customers.filter(
  F.col("customer_id").isNull() | (F.col("customer_id") == "") |
  F.col("country_code").isNull() | (F.col("country_code") == "") | (~F.col("country_code").isin(valid_country_codes)) | (F.length(F.col("country_code")) != 2)) \
  .withColumn("rejection_reason",
      F.when(F.col("customer_id").isNull() | (F.col("customer_id") == ""), "null/empty customer_id")
      .when(F.col("country_code").isNull() | (F.col("country_code") == ""), "null/empty country_code")
      .when(~F.col("country_code").isin(valid_country_codes), "invalid country_code")
      .when(F.length(F.col("country_code")) != 2, "invalid country_code length"))

In [0]:
# 6.2 Check Quarantined Data -> 

df_silver_customers_quarantine.select("customer_id", "country_code", "country_name", "state").show()

In [0]:
# 6.3 Write to Delta -> df_silver_customers_clean & df_silver_customers_quarantine

df_silver_customers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers_clean")

df_silver_customers_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers_quarantine")

In [0]:
display(df_silver_customers_clean)